# Rainbow line profiler — minimal notebook prototype

This notebook is the smallest possible prototype for checking the analysis loop in Jupyter.
It reuses the existing project code and starts with a fixed line on the demo image.

For the first pass, edit the line coordinates in the parameter cell and rerun the analysis cells.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
from PIL import Image
import plotly.graph_objects as go
from IPython.display import display

from src.config import load_config
from src.analysis.sampling import sample_line_rgb, compute_distances
from src.analysis.color import normalize_rgb, rgb_to_hsv_unwrapped
from src.analysis.rainbow import generate_reference_profile, compare_profiles

config = load_config()
demo_path = ROOT / config['demo']['path']
image = Image.open(demo_path).convert('RGB')
rgb = np.asarray(image)
h, w = rgb.shape[:2]
print(f'Loaded: {demo_path}')
print(f'Image size: {w} x {h}')


In [ ]:
# Line endpoints in image coordinates: (x, y)
start = (120, 180)
end = (980, 250)
linewidth = config['app']['default_line_width']

assert 0 <= start[0] < w and 0 <= start[1] < h
assert 0 <= end[0] < w and 0 <= end[1] < h
print('start =', start, 'end =', end, 'linewidth =', linewidth)


In [ ]:
fig = go.Figure()
fig.add_layout_image(dict(
    source=image,
    x=0, y=0,
    sizex=w, sizey=h,
    xref='x', yref='y',
    sizing='stretch',
    layer='below'
))
fig.add_trace(go.Scatter(
    x=[start[0], end[0]],
    y=[start[1], end[1]],
    mode='lines+markers',
    line=dict(color='cyan', width=3),
    marker=dict(size=8, color='cyan'),
    name='sample line'
))
fig.update_xaxes(range=[0, w], visible=False)
fig.update_yaxes(range=[h, 0], visible=False, scaleanchor='x')
fig.update_layout(width=min(w, 1100), height=min(h, 700), margin=dict(l=10, r=10, t=30, b=10), title='Demo image with sampling line')
fig.show()


In [ ]:
src = (start[1], start[0])
dst = (end[1], end[0])
sampled_rgb = sample_line_rgb(rgb, src=src, dst=dst, linewidth=linewidth)
distance = compute_distances(src, dst, len(sampled_rgb))
rgb01 = normalize_rgb(sampled_rgb)
hue, saturation, value = rgb_to_hsv_unwrapped(rgb01)
reference = generate_reference_profile(len(sampled_rgb), config)
comparison = compare_profiles(
    measured={'hue': hue, 'saturation': saturation},
    reference=reference,
)
comparison


In [ ]:
strip = np.repeat(rgb01[np.newaxis, :, :], 24, axis=0)
fig = go.Figure(go.Image(z=(strip * 255).astype(np.uint8)))
fig.update_layout(height=100, margin=dict(l=10, r=10, t=30, b=10), title='Sampled color strip')
fig.update_xaxes(showticklabels=False)
fig.update_yaxes(showticklabels=False)
fig.show()


In [ ]:
fig = go.Figure()
for i, color in enumerate(config['plots']['rgb_colors']):
    fig.add_trace(go.Scatter(x=distance, y=rgb01[:, i], mode='lines', name=['Red','Green','Blue'][i], line=dict(color=color)))
fig.update_layout(title='RGB channels along line', xaxis_title='Distance', yaxis_title='Intensity (0-1)')
fig.show()

fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=distance, y=hue, mode='lines', name='Measured hue', line=dict(color=config['plots']['hue_color'])))
fig2.add_trace(go.Scatter(x=distance, y=reference['hue'], mode='lines', name='Reference hue', line=dict(dash='dash')))
fig2.add_trace(go.Scatter(x=distance, y=saturation, mode='lines', name='Saturation', line=dict(color=config['plots']['sat_color'])))
fig2.update_layout(title='Hue and saturation along line', xaxis_title='Distance', yaxis_title='Value')
fig2.show()


## Next step ideas

- Replace fixed coordinates with sliders or draggable Plotly shapes.
- Add a small crop around the line for easier targeting.
- Wrap the analysis into one helper function for repeated use.
